<h1 style="font-size:40px; font-family: Verdana;">ALZHEIMER DESEASE</h1>

In [1]:
import torch
print(f"Versione PyTorch: {torch.__version__}")
print(f"CUDA disponibile? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")
else:
    print("ATTENZIONE: PyTorch non vede la GPU!")

Versione PyTorch: 2.5.1+cu121
CUDA disponibile? True
GPU rilevata: NVIDIA GeForce RTX 4090


#Part 1: Libraries

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch.nn.functional as F
import nibabel as nib
from sklearn.model_selection import StratifiedKFold
import pickle
import pandas as pd


#Part 2: Dataset Load and Normalization

In [3]:
#carico il file
with open("Dataset_Edin.pkl","rb") as f:
    data_dict=pickle.load(f) #definisco il data_dict

In [4]:

class AlzheimerDataset(Dataset):
    def __init__(self, fa_data, md_data, labels, target_shape=(64, 64, 64)):
        """
        fa_data: Array NumPy con tutti i volumi FA
        md_data: Array NumPy con tutti i volumi MD
        labels: Lista o array con le etichette
        """
        self.fa_data = fa_data
        self.md_data = md_data
        self.labels = labels
        self.target_shape = target_shape

    def __len__(self):
        return len(self.labels)

    def normalize(self, data):
        dmin, dmax = data.min(), data.max()
        if dmax - dmin > 0:
            return (data - dmin) / (dmax - dmin)
        return data

    def resize_volume(self, img_data):
        # Aggiungiamo dimensioni per PyTorch: [1, 1, D, H, W]
        img_tensor = torch.from_numpy(img_data).float().unsqueeze(0).unsqueeze(0)
        resized = F.interpolate(img_tensor, size=self.target_shape, mode='trilinear', align_corners=False)
        return resized.squeeze(0) # Ritorna [1, D, H, W]

    def __getitem__(self, idx):
        # Prendiamo il volume direttamente dall'array in memoria
        fa_vol = self.fa_data[idx]
        md_vol = self.md_data[idx]

        # Normalizzazione e Ridimensionamento
        fa_norm = self.resize_volume(self.normalize(np.nan_to_num(fa_vol)))
        md_norm = self.resize_volume(self.normalize(np.nan_to_num(md_vol)))

        # Concatenazione canali: [2, 64, 64, 64]
        x = torch.cat([fa_norm, md_norm], dim=0)
        y = torch.tensor(self.labels[idx], dtype=torch.long)

        return x, y


##3D CNN

In [5]:
class AlzheimerCNN3D(nn.Module):
    def __init__(self, num_classes=3):
        super(AlzheimerCNN3D, self).__init__()

        #Blocco 1: rileva le features più semplici (bordi, contrasti)
        self.conv1=nn.Conv3d(2, 32, kernel_size=3, padding=1)
        self.bn1=nn.BatchNorm3d(32)

        #Blocco 2: features più complesse
        self.conv2=nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.bn2=nn.BatchNorm3d(64)

        #Blocco 3: features profonde
        self.conv3=nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.bn3=nn.BatchNorm3d(128)

        self.pool=nn.MaxPool3d(2)
        self.dropout=nn.Dropout3d(0.3)

        #Parte Finale: classificazione
        #Se l'input è 64x64x64, dopo 3 pool(2^3) diventa 8x8x8
        self.fc1=nn.Linear(128*8*8*8, 256)
        self.fc2=nn.Linear(256, num_classes)

    def forward(self, x):
        #x shape: [Batch, 2, 64, 64, 64]
        x=self.pool(F.relu(self.bn1(self.conv1(x))))
        x=self.pool(F.relu(self.bn2(self.conv2(x))))
        x=self.pool(F.relu(self.bn3(self.conv3(x))))

        x=torch.flatten(x, 1)
        x=F.relu(self.fc1(x))
        x=self.dropout(x)
        x=self.fc2(x)
        return x


##FUNZIONE DI TRAINING

In [6]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss=0.0
    correct=0
    total=0

    for inputs, labels in loader:
        inputs, labels=inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs=model(inputs)
        loss=criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss+=loss.item()
        _, predicted=outputs.max(1)
        total+=labels.size(0)
        correct+=predicted.eq(labels).sum().item()

    return running_loss/len(loader), 100*correct/total

def validate_epoch(model, loader, criterion, device):
    model.eval()
    running_loss=0.0
    correct=0
    total=0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels= inputs.to(device), labels.to(device)
            labels=labels.long().view(-1)

            outputs=model(inputs)
            loss=criterion(outputs, labels)
            
            running_loss +=loss.item()
            _,predicted=outputs.max(1)

            total+=labels.size(0)
            #.item() assicura di estrarre il numero reale dal tensore
            correct+=predicted.eq(labels).sum().item()

    return running_loss/len(loader), 100*correct/total


##SCHEMA STRATIFIED K-FOLD

In [11]:
epochs=50
learning_rate=1e-5
k_folds=5
skf=StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Trasformo i dati del dizionario in array per lo split
X_fa=np.array(data_dict['FA'])
X_md=np.array(data_dict['MD'])
y=np.array(data_dict['Labels'])

fold_results=[] #Per salvare le performance di ogni Fold

for fold, (train_ids, val_ids) in enumerate (skf.split(X_fa, y)):
    print(f"\n---INIZIO FOLD {fold + 1}/{k_folds}---")

    # Creazione subset per questo fold
    train_ds = AlzheimerDataset(X_fa[train_ids], X_md[train_ids], y[train_ids])
    val_ds = AlzheimerDataset(X_fa[val_ids], X_md[val_ids], y[val_ids])
    
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
    
    # INIZIALIZZA MODELLO E OTTIMIZZATORE (Dentro il loop per resettarli!)
    model = AlzheimerCNN3D(num_classes=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    best_val_acc=0.0 #monitor la miglior acc per questo

    #LOOP delle EPOCHS
    for epoch in range(epochs):
        train_loss, train_acc=train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc=validate_epoch(model, val_loader, criterion, device)

        #best model checkpoint
        status=""
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            
        # STAMPA OGNI 10 EPOCHE
        if (epoch + 1) % 10 == 0:
            print(f"\nFold {fold+1} | Epoca {epoch+1:02d}: T-Loss: {train_loss:.3f} | T-Acc: {train_acc:.1f}% | V_loss:{val_loss:.3f} | V_Acc: {val_acc:.1f}%")
            if(epoch+1)%2==0: print("Continua", end="")
        elif (epoch + 1) % 2 == 0:
            print(".", end="", flush=True) # Mette un puntino per farti capire che sta lavorando
        
        
    fold_results.append(best_val_acc)
    print(f"\nMiglior Val Acc per il Fold {fold+1}: {best_val_acc:.2f}%")



---INIZIO FOLD 1/5---


/data/users/eibro/miniconda3/envs/torch_gpu/lib/python3.10/site-packages/torch/nn/functional.py:1597: UserWarning: dropout3d: Received a 2-D input to dropout3d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout3d exists to provide channel-wise dropout on inputs with 3 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 4D or 5D inputs).
  warnings.warn(warn_msg)


....
Fold 1 | Epoca 10: T-Loss: 0.907 | T-Acc: 52.8% | V_loss:1.044 | V_Acc: 48.7%
Continua....
Fold 1 | Epoca 20: T-Loss: 0.679 | T-Acc: 67.8% | V_loss:1.043 | V_Acc: 48.7%
Continua....
Fold 1 | Epoca 30: T-Loss: 0.570 | T-Acc: 68.1% | V_loss:1.020 | V_Acc: 50.0%
Continua....
Fold 1 | Epoca 40: T-Loss: 0.391 | T-Acc: 77.4% | V_loss:1.030 | V_Acc: 52.6%
Continua....
Fold 1 | Epoca 50: T-Loss: 0.350 | T-Acc: 78.4% | V_loss:1.046 | V_Acc: 53.9%
ContinuaMiglior Val Acc per il Fold 1: 55.26%

---INIZIO FOLD 2/5---
....
Fold 2 | Epoca 10: T-Loss: 0.951 | T-Acc: 57.1% | V_loss:1.027 | V_Acc: 43.4%
Continua....
Fold 2 | Epoca 20: T-Loss: 0.715 | T-Acc: 69.8% | V_loss:1.024 | V_Acc: 46.1%
Continua....
Fold 2 | Epoca 30: T-Loss: 0.501 | T-Acc: 84.4% | V_loss:1.019 | V_Acc: 46.1%
Continua....
Fold 2 | Epoca 40: T-Loss: 0.458 | T-Acc: 84.7% | V_loss:1.013 | V_Acc: 47.4%
Continua....
Fold 2 | Epoca 50: T-Loss: 0.380 | T-Acc: 85.4% | V_loss:1.043 | V_Acc: 46.1%
ContinuaMiglior Val Acc per il Fold 2

In [12]:
#REPORT FINALE
print(f"\n{'#'*40}")
print(f"ACCURATEZZA MEDIA K-FOLD: {np.mean(fold_results):.2f}%")
print(f"{'#'*40}")


########################################
ACCURATEZZA MEDIA K-FOLD: 57.85%
########################################
